# Transform BOU_Transactions Data

1. Read bronze `transactions` table
1. Drop unwanted columns
1. Standardise column names using snake_case 
1. Filter out rows where `txn_id` is null (business key validation)
1. Remove duplicate records
1. Write the transformed data to silver `circuits` table

> Below changes are required to implement Incremental Load Processing
1. Accept batch_date as a parameter to the notebook
1. Process data for only the batch_date being passed in (i.e., filter reading from bronze using the batch_date)
1. Add created_timestamp, updated_timestamp and batch_date to the silver table. 
1. Merge the processed data to the silver table
    - created_timestamp should only be populated at the time of inserting/ creating the record. It should not be updated during the merge update.
    - Ensure that we are not overwriting the data in silver table by older bronze data (re-run scenario)

In [0]:
dbutils.widgets.text("p_batch_date","")
v_batch_date = dbutils.widgets.get("p_batch_date")

In [0]:
import pyspark.sql.functions as F

In [0]:
%run ../00-common/01.environment_config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.transactions"
silver_table = f"{catalog_name}.{silver_schema}.transactions"

In [0]:
txn_df = spark.read.table("payment_app.bronze.transactions").filter(F.col("batch_id")==v_batch_date)

In [0]:
display(txn_df)

TxnID,AccountNumber,BillerID,ConsumerNumber,TxnAmount,Status,RRN,BillerRefID,TransactionDate,ingestion_timestamp,source_file,batch_id
TXN1001,100001,KSEB,CONS001,1500.00,Success,RRN100001,BREF100001,2026-08-07T09:05:00.000Z,2026-08-24T09:42:11.500Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/transactions/data_7a084b7a-22ee-4428-8089-597b981a17ac_d095ba69-a0b7-46bc-8e25-c26a7624f195.parquet,2026-08-13
TXN1002,100002,WTR01,CONS002,800.00,Success,RRN100002,BREF100002,2026-08-07T09:15:00.000Z,2026-08-24T09:42:11.500Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/transactions/data_7a084b7a-22ee-4428-8089-597b981a17ac_d095ba69-a0b7-46bc-8e25-c26a7624f195.parquet,2026-08-13
TXN1003,100003,TEL01,CONS003,999.00,Pending,null,null,2026-08-07T09:30:00.000Z,2026-08-24T09:42:11.500Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/transactions/data_7a084b7a-22ee-4428-8089-597b981a17ac_d095ba69-a0b7-46bc-8e25-c26a7624f195.parquet,2026-08-13
TXN1004,100004,GAS01,CONS004,1200.00,Failed,null,null,2026-08-07T09:40:00.000Z,2026-08-24T09:42:11.500Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/transactions/data_7a084b7a-22ee-4428-8089-597b981a17ac_d095ba69-a0b7-46bc-8e25-c26a7624f195.parquet,2026-08-13
TXN1005,100005,DTH01,CONS005,450.00,Success,RRN100005,BREF100005,2026-08-07T09:50:00.000Z,2026-08-24T09:42:11.500Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/transactions/data_7a084b7a-22ee-4428-8089-597b981a17ac_d095ba69-a0b7-46bc-8e25-c26a7624f195.parquet,2026-08-13
TXN1006,100006,KSEB,CONS006,2300.00,Success,RRN100006,BREF100006,2026-08-07T10:05:00.000Z,2026-08-24T09:42:11.500Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/transactions/data_7a084b7a-22ee-4428-8089-597b981a17ac_d095ba69-a0b7-46bc-8e25-c26a7624f195.parquet,2026-08-13
TXN1007,100007,GAS01,CONS007,1750.00,Success,RRN100007,BREF100007,2026-08-07T10:20:00.000Z,2026-08-24T09:42:11.500Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/transactions/data_7a084b7a-22ee-4428-8089-597b981a17ac_d095ba69-a0b7-46bc-8e25-c26a7624f195.parquet,2026-08-13
TXN1008,100008,TEL01,CONS008,699.00,Success,RRN100008,BREF100008,2026-08-07T10:35:00.000Z,2026-08-24T09:42:11.500Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/transactions/data_7a084b7a-22ee-4428-8089-597b981a17ac_d095ba69-a0b7-46bc-8e25-c26a7624f195.parquet,2026-08-13
TXN1009,100009,WTR01,CONS009,650.00,Pending,null,null,2026-08-07T10:45:00.000Z,2026-08-24T09:42:11.500Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/transactions/data_7a084b7a-22ee-4428-8089-597b981a17ac_d095ba69-a0b7-46bc-8e25-c26a7624f195.parquet,2026-08-13
TXN1010,100010,DTH01,CONS010,399.00,Success,RRN100010,BREF100010,2026-08-07T11:00:00.000Z,2026-08-24T09:42:11.500Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/transactions/data_7a084b7a-22ee-4428-8089-597b981a17ac_d095ba69-a0b7-46bc-8e25-c26a7624f195.parquet,2026-08-13


In [0]:
#technical transformations

txn_std= txn_df.withColumnsRenamed({"TxnID":"txn_id","AccountNumber": "account_number","BillerID":"biller_id","ConsumerNumber":"consumer_number","TxnAmount":"txn_amount","Status":"status","BillerRefID":"biller_ref_id","TransactionDate":"transaction_date"})

In [0]:
#data quality checks

#null check
null_check = (
    F.col("txn_id").isNotNull() &
    F.col("account_number").isNotNull() &
    F.col("biller_id").isNotNull() &
    F.col("consumer_number").isNotNull() &
    F.col("txn_amount").isNotNull() &
    F.col("status").isNotNull() &
    F.col("transaction_date").isNotNull()
)

txn_valid = txn_std.filter(null_check)

txn_rejected = txn_std.filter(~null_check)


txn_rejected = txn_rejected.withColumn(
    "reject_reason",
    F.lit("Mandatory field is NULL")
)

txn_rejected.write.format("delta").mode("append").saveAsTable(
    "silver_transaction_rejects"
)

In [0]:
#data quality checks

#de-dupe - reject duplicates

txn_deduped = txn_valid.dropDuplicates(["txn_id"])

##If you also want to quarantine  the extra copies, rather than reject all other instances, then we need row_number() and keep rn = 1. That's the better approach when you want both a clean Silver dataset and a precise reject set.

In [0]:
#ref-integrity check - need biller_master df

txn_valid_billers = txn_deduped.join(
    billers.select("biller_id").distinct(),
    on="biller_id",
    how="left_semi"
)

In [0]:
#domain check

valid_status = ["Success", "Failed", "Pending"]

invalid_status = txn_deduped.filter(
    ~F.col("status").isin(valid_status)
)

txn_status_valid = txn_deduped.filter(
    F.col("status").isin(valid_status)
)

invalid_status.write.format("delta").mode("append").saveAsTable(
    "silver_transaction_rejects"

In [0]:
#range check

invalid_amount = txn_valid.filter(
    F.col("txn_amount") <= 0
)

invalid_amount = invalid_amount.withColumn(
    "reject_reason",
    F.lit("Invalid transaction amount")
)

invalid_amount.write.format("delta").mode("append").saveAsTable(
    "silver_transaction_rejects"
)

txn_valid = txn_valid.filter(
    F.col("txn_amount") > 0
)

In [0]:
"""
business tranformations

Enrich with Biller attributes
Join the transaction with the Biller master to bring required business attributes into Silver, such as biller category or biller name—again, only if your Silver model calls for them.
"""

In [0]:
silver_cols = [
    "txn_id",
    "account_number",
    "biller_id",
    "consumer_number",
    "txn_amount",
    "status",
    "rrn",
    "biller_ref_id",
    "transaction_date",
    "batch_id"
]

txn_silver = txn_valid.select(silver_cols)

In [0]:
display(txn_silver)

txn_id,account_number,biller_id,consumer_number,txn_amount,status,rrn,biller_ref_id,transaction_date,batch_id
TXN1001,100001,KSEB,CONS001,1500.00,Success,RRN100001,BREF100001,2026-08-07T09:05:00.000Z,2026-08-13
TXN1002,100002,WTR01,CONS002,800.00,Success,RRN100002,BREF100002,2026-08-07T09:15:00.000Z,2026-08-13
TXN1003,100003,TEL01,CONS003,999.00,Pending,null,null,2026-08-07T09:30:00.000Z,2026-08-13
TXN1004,100004,GAS01,CONS004,1200.00,Failed,null,null,2026-08-07T09:40:00.000Z,2026-08-13
TXN1005,100005,DTH01,CONS005,450.00,Success,RRN100005,BREF100005,2026-08-07T09:50:00.000Z,2026-08-13
TXN1006,100006,KSEB,CONS006,2300.00,Success,RRN100006,BREF100006,2026-08-07T10:05:00.000Z,2026-08-13
TXN1007,100007,GAS01,CONS007,1750.00,Success,RRN100007,BREF100007,2026-08-07T10:20:00.000Z,2026-08-13
TXN1008,100008,TEL01,CONS008,699.00,Success,RRN100008,BREF100008,2026-08-07T10:35:00.000Z,2026-08-13
TXN1009,100009,WTR01,CONS009,650.00,Pending,null,null,2026-08-07T10:45:00.000Z,2026-08-13
TXN1010,100010,DTH01,CONS010,399.00,Success,RRN100010,BREF100010,2026-08-07T11:00:00.000Z,2026-08-13


In [0]:
silver_final = txn_silver.withColumn("created_timestamp", F.current_timestamp()) \
                    .withColumn("updated_timestamp", F.current_timestamp())

In [0]:
silver_final.createOrReplaceTempView("vw_silver_final")

if not spark.catalog.tableExists(silver_table):

    (
        silver_final.write
        .format("delta")
        .partitionBy("batch_id", "biller_id")
        .mode("overwrite")
        .saveAsTable(silver_table)
    )

else:

    spark.sql(f"""
    MERGE INTO {silver_table} AS tgt
    USING vw_silver_final AS src
    ON tgt.txn_id = src.txn_id

    WHEN MATCHED
    AND src.batch_date >= tgt.batch_date
    THEN UPDATE SET
        tgt.account_number   = src.account_number,
        tgt.biller_id        = src.biller_id,
        tgt.consumer_number  = src.consumer_number,
        tgt.txn_amount       = src.txn_amount,
        tgt.status            = src.status,
        tgt.rrn               = src.rrn,
        tgt.biller_ref_id    = src.biller_ref_id,
        tgt.transaction_date = src.transaction_date,
        tgt.batch_id       = src.batch_id,
        tgt.updated_timestamp = current_timestamp()

    WHEN NOT MATCHED
    THEN INSERT (
        txn_id,
        account_number,
        biller_id,
        consumer_number,
        txn_amount,
        status,
        rrn,
        biller_ref_id,
        transaction_date,
        batch_id,
        created_timestamp,
        updated_timestamp
    )
    VALUES (
        src.txn_id,
        src.account_number,
        src.biller_id,
        src.consumer_number,
        src.txn_amount,
        src.status,
        src.rrn,
        src.biller_ref_id,
        src.transaction_date,
        src.batch_id,
        src.created_timestamp,
        src.updated_timestamp
    )
    """)

In [0]:
input_count = txn_std.count()
valid_count = txn_valid.count()
rejected_count = rejected_df.count()

if input_count != valid_count + rejected_count:
    raise Exception(
        f"Reconciliation failed: "
        f"input={input_count}, "
        f"valid={valid_count}, "
        f"rejected={rejected_count}"
    )

In [0]:
from pyspark.sql import Row
from pyspark.sql import functions as F

audit_df = spark.createDataFrame([
    Row(
        batch_date=batch_date,
        input_count=input_count,
        valid_count=valid_count,
        rejected_count=rejected_count,
        processing_timestamp=None,
        status="SUCCESS"
    )
]).withColumn(
    "processing_timestamp",
    F.current_timestamp()
)

In [0]:
#if audit table is there in adf  ??